# PRISM-EEG — Periodicity-Rhythm Images with Symmetric hemispheric Modelling

A new model for HMS, designed from the ACNS definitions of the six classes (see `prism.py` docstring and figures):

1. **RPS image per bipolar channel**, computed on the GPU:
   - **R** — signal autocorrelation (rhythm)
   - **P** — envelope autocorrelation (periodicity of discharges)
   - **S** — log spectrum 0.5–20 Hz
2. **Symmetric hemispheric encoder:** one shared pretrained CNN for the left and mirrored right chains, fused as (L+R)/2 and |L−R|, plus a midline stream. The output is exactly invariant to hemisphere swap, polarity and amplitude.
3. **Evolution transformer** over 8 time tokens and their differences (seizure evolution).
4. **ACNS-factorised head:** P(Seizure, PD, RDA, Other) × P(lateralized | PD) × P(lateralized | RDA).

**Shared with existing solutions (data handling only):**
- VIPEEGNet's EEG cache, row table, patient split and validation windows, so scores are paired row by row with the archived baseline
- ±5 s window jitter
- two-stage schedule (all rows weighted votes/20, then ≥10-vote rows)
- an ImageNet-pretrained CNN backbone

**Fair reporting:**
- No checkpoint is chosen on validation. The final EMA weights of every fold are used.
- Outputs go to `fair_comparison/prism_<tag>/` (no TTA) and `…_tta/` (time-reversal TTA), in the archive schema.

**Ablation switches:**
- `HEAD='flat'` (plain 6-way softmax)
- `FUSION='concat'` (no symmetric fusion)
- `REPR='spec'` (spectrum only, no periodicity maps)
- `LOSS='kl+dm'` (adds a Dirichlet-multinomial vote likelihood in stage 1)

## Parameters

In [1]:
PROJECT_DIR = '~/hms'
TAG = 'v1'
BACKBONE = 'efficientnet_b0'      # any timm model; e.g. 'convnext_nano', 'efficientnet_b2'
HEAD, FUSION, REPR, LOSS = 'acns', 'sym', 'wave+acf', 'kl'   # REPR: 'wave+acf' (v4) | 'wave' | 'rps+wave' (v3) | 'rps' (v2) | 'spec'
WAVE_WIDTH = 1.0                    # channel multiplier of the waveform encoder
NORM, SPECAUG = 'global', True     # v2: keep cross-channel power; time masking. v1 = 'channel', False
EPOCHS1, EPOCHS2 = 12, 5            # waveform models were still improving at epoch 6 (image models overfit)
LR1, LR2, HEAD_LR_MULT = 3e-4, 5e-5, 3.0
BATCH, WORKERS = 32, 8
FOLDS = [0, 1, 2, 3, 4]
SMOKE_TEST = False
SEED = 2024
PRETRAINED = True                 # ImageNet weights via timm (downloads once)

In [2]:
# Parameters
FOLDS = [0]
PROJECT_DIR = "/home/nfs/jhb-dehzangi/ws04/gs1147/hms"
TAG = "w_flat"
REPR = "wave"
HEAD = "flat"


In [3]:
import os, sys, json, time
import numpy as np, pandas as pd, torch
P = os.path.abspath(os.path.expanduser(PROJECT_DIR))
sys.path.insert(0, P)
from prism_train import load_tables, load_eegs, run_fold, kld_rows, TARGETS
from prism import CLASSES
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
torch.backends.cudnn.benchmark = True
if SMOKE_TEST:
    EPOCHS1, EPOCHS2, FOLDS = 2, 1, [0]
RUN = f'prism_{TAG}' + ('_smoke' if SMOKE_TEST else '')
OUT, WDIR = f'{P}/fair_comparison/{RUN}', f'{P}/prism/output_{RUN}'
for d in (OUT, WDIR): os.makedirs(d, exist_ok=True)
SPLIT = f'{P}/results/vipeegnet_groupkfold_v1/folds/patient_folds_groupkfold.csv'
cfg = dict(backbone=BACKBONE, pretrained=PRETRAINED, head=HEAD, fusion=FUSION, repr=REPR, loss=LOSS, dm_weight=0.5,
           norm=NORM, specaug=SPECAUG, wave_width=WAVE_WIDTH,
           epochs1=EPOCHS1, epochs2=EPOCHS2, lr1=LR1, lr2=LR2, head_lr_mult=HEAD_LR_MULT, wd=0.02, clip=2.0,
           ema=0.998, batch=BATCH, workers=WORKERS, seed=SEED)
print(RUN, json.dumps(cfg))

torch 2.14.1+cu130 | device cuda NVIDIA RTX A6000
prism_w_flat {"backbone": "efficientnet_b0", "pretrained": true, "head": "flat", "fusion": "sym", "repr": "wave", "loss": "kl", "dm_weight": 0.5, "norm": "global", "specaug": true, "wave_width": 1.0, "epochs1": 12, "epochs2": 5, "lr1": 0.0003, "lr2": 5e-05, "head_lr_mult": 3.0, "wd": 0.02, "clip": 2.0, "ema": 0.998, "batch": 32, "workers": 8, "seed": 2024}


In [4]:
rows, lookup = load_tables(f'{P}/hms-data', SPLIT, smoke=SMOKE_TEST)
eegs = load_eegs(f'{P}/eeg_cache_v2', rows.eeg_id.unique())
print(f'{len(rows):,} rows | {rows.patient_id.nunique()} patients | >=10 votes: {(rows.votes >= 10).sum():,}')
for k in range(5):
    r = rows[rows.fold == k]; print(f'  fold {k}: {len(r):,} rows, {(r.votes >= 10).sum():,} with >=10 votes')

/home/nfs/jhb-dehzangi/ws04/gs1147/hms/.venv-torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load EEG cache:   0%|          | 0/17089 [00:00<?, ?it/s]

load EEG cache:   0%|          | 58/17089 [00:00<00:29, 570.65it/s]

load EEG cache:   1%|          | 116/17089 [00:00<00:38, 441.21it/s]

load EEG cache:   1%|          | 162/17089 [00:00<00:39, 428.96it/s]

load EEG cache:   1%|          | 206/17089 [00:00<00:39, 424.28it/s]

... (376 outputs left out)

load EEG cache: 100%|█████████▉| 17025/17089 [00:39<00:00, 437.56it/s]

load EEG cache: 100%|█████████▉| 17069/17089 [00:39<00:00, 423.91it/s]

load EEG cache: 100%|██████████| 17089/17089 [00:39<00:00, 427.75it/s]

20,183 rows | 1950 patients | >=10 votes: 6,350
  fold 0: 4,037 rows, 1,376 with >=10 votes
  fold 1: 4,037 rows, 1,228 with >=10 votes
  fold 2: 4,037 rows, 1,229 with >=10 votes
  fold 3: 4,036 rows, 1,155 with >=10 votes
  fold 4: 4,036 rows, 1,362 with >=10 votes


## Train + out-of-fold prediction

In [5]:
results = []
t_all = time.time()
for k in FOLDS:
    t0 = time.time()
    res = run_fold(k, rows, lookup, eegs, cfg, OUT, WDIR, device=DEVICE)
    results.append(res)
    print(f'FOLD {k}: KLD >=10 votes {res["kld_hq"]:.4f} (TTA {res["kld_hq_tta"]:.4f}) | all rows {res["kld_all"]:.4f}'
          f' | {(time.time() - t0) / 60:.1f} min', flush=True)
    json.dump(results, open(f'{WDIR}/results_partial.json', 'w'), indent=1)
print(f'done in {(time.time() - t_all) / 3600:.2f} h')

/home/nfs/jhb-dehzangi/ws04/gs1147/hms/.venv-torch/lib/python3.10/site-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(


/home/nfs/jhb-dehzangi/ws04/gs1147/hms/prism.py:205: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tx = nn.TransformerEncoder(layer, n_layers)


  fold 0 stage1 ep 1/12  train 0.2498  val>=10 (EMA) 0.4774  68s


  fold 0 stage1 ep 2/12  train 0.1802  val>=10 (EMA) 0.3749  56s


  fold 0 stage1 ep 3/12  train 0.1604  val>=10 (EMA) 0.3405  57s


  fold 0 stage1 ep 4/12  train 0.1517  val>=10 (EMA) 0.3111  57s


... (11 outputs left out)

  fold 0 stage2 ep 4/5  train 0.2286  val>=10 (EMA) 0.2707  19s


  fold 0 stage2 ep 5/5  train 0.2233  val>=10 (EMA) 0.2696  19s


FOLD 0: KLD >=10 votes 0.2696 (TTA 0.2700) | all rows 0.6259 | 13.6 min


done in 0.23 h


## Summary (same rows as the VIPEEGNet archive)

In [6]:
def pooled(d, sub):
    yt, yp = [], []
    for k in FOLDS:
        z = np.load(f'{d}/fold{k}_{sub}.npz'); yt.append(z['yt']); yp.append(z['yp'])
    return float(kld_rows(np.concatenate(yt), np.concatenate(yp)).mean()), np.concatenate(yt), np.concatenate(yp)
summ = {'model': 'PRISM-EEG', 'run': RUN, 'cfg': cfg, 'per_fold': results, 'fair_splits': 'VIPEEGNet patient split'}
print('=' * 66)
for tag in ('', '_tta'):
    hq, yt_h, yp_h = pooled(OUT + tag, 'oof'); al, _, _ = pooled(OUT + tag, 'oof_all')
    mf = float(np.mean([r[f'kld_hq{tag}'] for r in results]))
    summ[f'oof_kld_hq{tag or "_native"}'] = hq; summ[f'mean_fold_kld_hq{tag}'] = mf; summ[f'oof_kld_all{tag}'] = al
    print(f'  PRISM {"+ TTA" if tag else "     "}: pooled >=10 votes {hq:.4f} | mean of folds {mf:.4f} | all rows {al:.4f}')
print('  VIPEEGNet       : pooled >=10 votes 0.2217 | mean of folds 0.2221')
print('  per fold (no TTA):', [round(r['kld_hq'], 4) for r in results])
print('=' * 66)
summ['oof_kld_hq_raw'] = summ['oof_kld_hq_native']; summ['n_oof_hq'] = int(sum(r['n_valid_hq'] for r in results))
summ['recipe'] = f'PRISM {HEAD}/{FUSION}/{REPR}/{LOSS} {BACKBONE} {EPOCHS1}+{EPOCHS2} ep'; summ['tta'] = 'none'
for d in (OUT, OUT + '_tta'):
    json.dump(summ, open(f'{d}/summary.json', 'w'), indent=1, default=float)

  PRISM      : pooled >=10 votes 0.2696 | mean of folds 0.2696 | all rows 0.6259
  PRISM + TTA: pooled >=10 votes 0.2700 | mean of folds 0.2700 | all rows 0.6220
  VIPEEGNet       : pooled >=10 votes 0.2217 | mean of folds 0.2221
  per fold (no TTA): [0.2696]


In [7]:
from sklearn.metrics import roc_auc_score, confusion_matrix
_, yt, yp = pooled(OUT, 'oof')
yti, ypi = yt.argmax(1), yp.argmax(1); oh = (yti[:, None] == np.arange(6)).astype(int)
cm = confusion_matrix(yti, ypi, labels=range(6))
with np.errstate(invalid='ignore', divide='ignore'):
    rec, prec = np.diag(cm) / cm.sum(1) * 100, np.diag(cm) / cm.sum(0) * 100
vip = {'AUROC': [0.975, 0.963, 0.976, 0.943, 0.952, 0.929]}
print(f'{"class":>8} | AUROC (VIPEEGNet) | recall | precision')
for c in range(6):
    print(f'{CLASSES[c]:>8} | {roc_auc_score(oh[:, c], yp[:, c]):.3f} ({vip["AUROC"][c]:.3f})   | {rec[c]:5.1f}% | {prec[c]:5.1f}%')

   class | AUROC (VIPEEGNet) | recall | precision
 Seizure | 0.967 (0.975)   |  37.9% |  64.7%
     LPD | 0.935 (0.963)   |  68.4% |  79.6%
     GPD | 0.976 (0.976)   |  73.5% |  74.0%
    LRDA | 0.903 (0.943)   |  21.3% |  57.4%
    GRDA | 0.926 (0.952)   |  36.1% |  40.0%
   Other | 0.912 (0.929)   |  89.1% |  70.0%
